# Grok-rl-01-bandits

**领域**: 强化学习 From Scratch — Stage 01  
**环境**: Kaggle T4×2（本阶段主要为 CPU 数值实验；GPU 仅做可用性记录）

## 概念
在**不知道哪只手臂回报最高**时，如何在 **探索(exploration)** 与 **利用(exploitation)** 之间权衡？

多臂老虎机 (Multi-Armed Bandit) 是 RL 的零基础：没有状态转移，只有动作→奖励。

## 本阶段算法
1. ε-greedy
2. UCB1
3. Thompson Sampling (Bernoulli)

## 观察指标
- 累积遗憾 (cumulative regret)
- 平均奖励曲线
- 最优臂选择比例


In [ ]:

import json, math, os, time, platform
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
rng = np.random.default_rng(SEED)

# GPU probe (optional this stage)
gpu_info = {"cuda": False, "device_count": 0, "names": []}
try:
    import torch
    gpu_info["cuda"] = bool(torch.cuda.is_available())
    gpu_info["device_count"] = int(torch.cuda.device_count()) if gpu_info["cuda"] else 0
    gpu_info["names"] = [torch.cuda.get_device_name(i) for i in range(gpu_info["device_count"])]
    gpu_info["torch"] = torch.__version__
except Exception as e:
    gpu_info["error"] = str(e)

print("GPU:", gpu_info)
print("numpy", np.__version__, "python", platform.python_version())


## 最小环境：Bernoulli Bandit

真实输入：每条臂有未知成功率 `p_a`。拉臂得到 0/1 奖励。


In [ ]:

class BernoulliBandit:
    """K-armed Bernoulli bandit. True means p are fixed; agent never sees them."""
    def __init__(self, probs, rng):
        self.probs = np.asarray(probs, dtype=np.float64)
        self.K = len(self.probs)
        self.rng = rng
        self.opt = float(self.probs.max())
        self.opt_arm = int(self.probs.argmax())

    def pull(self, a: int) -> float:
        return float(self.rng.random() < self.probs[a])

# Real input: 10 arms, clearly separated optimum
TRUE_P = np.array([0.10, 0.20, 0.25, 0.15, 0.55, 0.35, 0.30, 0.40, 0.22, 0.18])
env = BernoulliBandit(TRUE_P, rng)
print("true p:", TRUE_P)
print("optimal arm", env.opt_arm, "p*=", env.opt)


## 算法实现


In [ ]:

class EpsilonGreedy:
    def __init__(self, K, eps, rng):
        self.K, self.eps, self.rng = K, eps, rng
        self.n = np.zeros(K)
        self.q = np.zeros(K)

    def act(self):
        if self.rng.random() < self.eps:
            return int(self.rng.integers(0, self.K))
        # break ties randomly
        m = self.q.max()
        cands = np.flatnonzero(np.isclose(self.q, m))
        return int(self.rng.choice(cands))

    def update(self, a, r):
        self.n[a] += 1
        self.q[a] += (r - self.q[a]) / self.n[a]


class UCB1:
    def __init__(self, K, c=2.0):
        self.K, self.c = K, c
        self.n = np.zeros(K)
        self.q = np.zeros(K)
        self.t = 0

    def act(self):
        self.t += 1
        # pull each once
        for a in range(self.K):
            if self.n[a] == 0:
                return a
        bonus = self.c * np.sqrt(np.log(self.t) / self.n)
        u = self.q + bonus
        return int(np.argmax(u))

    def update(self, a, r):
        self.n[a] += 1
        self.q[a] += (r - self.q[a]) / self.n[a]


class ThompsonSampling:
    def __init__(self, K, rng):
        self.K, self.rng = K, rng
        self.alpha = np.ones(K)  # Beta(1,1) prior
        self.beta = np.ones(K)

    def act(self):
        samples = self.rng.beta(self.alpha, self.beta)
        return int(np.argmax(samples))

    def update(self, a, r):
        if r > 0.5:
            self.alpha[a] += 1
        else:
            self.beta[a] += 1


def run_agent(factory, env, steps, seed):
    local = np.random.default_rng(seed)
    # re-seed env rng for fair comparison paths... keep env stochastic independent
    agent = factory(local)
    rewards = np.zeros(steps)
    opt_hits = np.zeros(steps)
    regret = np.zeros(steps)
    cum_r = 0.0
    cum_reg = 0.0
    for t in range(steps):
        a = agent.act()
        r = env.pull(a)
        agent.update(a, r)
        cum_r += r
        cum_reg += (env.opt - env.probs[a])
        rewards[t] = cum_r / (t + 1)
        opt_hits[t] = 1.0 if a == env.opt_arm else 0.0
        regret[t] = cum_reg
    # smooth opt rate
    opt_rate = np.cumsum(opt_hits) / (np.arange(steps) + 1)
    return {"avg_reward": rewards, "cum_regret": regret, "opt_rate": opt_rate}


## 实验：同一环境，对比三种策略


In [ ]:

STEPS = 5000
N_RUNS = 30
K = len(TRUE_P)

def make_factories():
    return {
        "eps-greedy(0.1)": lambda rng: EpsilonGreedy(K, 0.1, rng),
        "eps-greedy(0.01)": lambda rng: EpsilonGreedy(K, 0.01, rng),
        "UCB1(c=2)": lambda rng: UCB1(K, c=2.0),
        "Thompson": lambda rng: ThompsonSampling(K, rng),
    }

# multi-run average
results = {}
t0 = time.time()
for name, fac in make_factories().items():
    acc = {"avg_reward": np.zeros(STEPS), "cum_regret": np.zeros(STEPS), "opt_rate": np.zeros(STEPS)}
    for run in range(N_RUNS):
        env_run = BernoulliBandit(TRUE_P, np.random.default_rng(10_000 + run))
        out = run_agent(fac, env_run, STEPS, seed=2000 + run)
        for k in acc:
            acc[k] += out[k]
    for k in acc:
        acc[k] /= N_RUNS
    results[name] = acc
    print(f"{name:18s} final_avgR={acc['avg_reward'][-1]:.3f}  regret={acc['cum_regret'][-1]:.1f}  opt%={acc['opt_rate'][-1]*100:.1f}")

elapsed = time.time() - t0
print(f"elapsed {elapsed:.2f}s")


In [ ]:

# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for name, acc in results.items():
    axes[0].plot(acc["avg_reward"], label=name)
    axes[1].plot(acc["cum_regret"], label=name)
    axes[2].plot(acc["opt_rate"], label=name)
axes[0].axhline(TRUE_P.max(), ls="--", c="k", alpha=0.4, label="optimal mean")
axes[0].set_title("Average reward"); axes[0].set_xlabel("t"); axes[0].legend(fontsize=8)
axes[1].set_title("Cumulative regret"); axes[1].set_xlabel("t"); axes[1].legend(fontsize=8)
axes[2].set_title("Optimal arm rate"); axes[2].set_xlabel("t"); axes[2].legend(fontsize=8)
fig.suptitle("Stage 01 Bandits — exploration strategies")
fig.tight_layout()
fig_path = OUT / "stage01_bandits_curves.png"
fig.savefig(fig_path, dpi=120)
print("saved", fig_path)
plt.close(fig)

# Final comparison table
summary = []
for name, acc in results.items():
    summary.append({
        "agent": name,
        "final_avg_reward": float(acc["avg_reward"][-1]),
        "final_cum_regret": float(acc["cum_regret"][-1]),
        "final_opt_rate": float(acc["opt_rate"][-1]),
    })
summary = sorted(summary, key=lambda x: x["final_cum_regret"])
print(json.dumps(summary, indent=2))


## 与“零策略”对比

随机乱拉 = 纯探索；纯 greedy 在错误先验下会永久卡住。  
**新增能力**：用不确定性（UCB 置信上界 / 后验采样）系统化探索。


In [ ]:

# Random baseline for contrast
rand_regrets = []
for run in range(N_RUNS):
    env_run = BernoulliBandit(TRUE_P, np.random.default_rng(3000+run))
    cum = 0.0
    for t in range(STEPS):
        a = int(rng.integers(0, K))
        r = env_run.pull(a)
        cum += env_run.opt - env_run.probs[a]
    rand_regrets.append(cum)
random_regret = float(np.mean(rand_regrets))

best = summary[0]
payload = {
    "ok": True,
    "stage": "01-bandits",
    "title": "Grok-rl-01-bandits",
    "seed": SEED,
    "steps": STEPS,
    "n_runs": N_RUNS,
    "true_probs": TRUE_P.tolist(),
    "optimal_arm": int(env.opt_arm),
    "summary": summary,
    "random_baseline_cum_regret": random_regret,
    "best_agent": best["agent"],
    "improvement_vs_random_regret_ratio": float(random_regret / max(best["final_cum_regret"], 1e-9)),
    "gpu": gpu_info,
    "elapsed_sec": elapsed,
    "concept": "exploration-exploitation tradeoff without state transitions",
    "new_capability": "structured exploration (UCB/Thompson) reduces cumulative regret vs random/epsilon",
    "compare_to_previous": "Stage 00 had no decision problem; now we learn which action is best from rewards alone",
}
(OUT / "results_stage01.json").write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2)[:1200])
assert payload["ok"]
assert best["final_cum_regret"] < random_regret * 0.7, "best agent should clearly beat random"
print("STAGE01_OK")
